# 03 — Ablation Studies

Runs the 6-variant ablation matrix (A1–A6) and produces the comparison plot for the report.

**Quick mode** (default below) uses a smaller number of epochs so you can run all six in ~1 hour on free Kaggle GPU. For final results, bump `cfg.training.num_epochs` back up.

In [ ]:
import sys, os, json
from pathlib import Path
sys.path.insert(0, '..')

from src.evaluation.ablation import ABLATIONS, run_one
print('Ablation variants:')
for k in ABLATIONS:
    print(f'  • {k}')

In [ ]:
# Quick mode: override num_epochs down
QUICK = True
extra_overrides = ['training.num_epochs=4'] if QUICK else []

results = {}
for key, overrides in ABLATIONS.items():
    print(f'\n========== Running {key} ==========')
    try:
        metrics = run_one('../src/config/config.yaml', overrides + extra_overrides)
        results[key] = metrics
    except Exception as e:
        print(f'!! {key} failed: {e}')
        results[key] = {'error': str(e)}

with open('../docs/ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

In [ ]:
# Visualize
import matplotlib.pyplot as plt
import numpy as np

keys = [k for k in results if 'error' not in results[k]]
s_acc = [results[k].get('stress/balanced_acc', 0) for k in keys]
f_acc = [results[k].get('fatigue/balanced_acc', 0) for k in keys]

x = np.arange(len(keys))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 0.2, s_acc, 0.4, label='Stress balanced acc')
ax.bar(x + 0.2, f_acc, 0.4, label='Fatigue balanced acc')
ax.set_xticks(x)
ax.set_xticklabels([k.split('_', 1)[0] for k in keys])
ax.set_ylabel('Balanced accuracy')
ax.set_title('Ablation results')
ax.legend()
plt.tight_layout()
plt.savefig('../assets/screenshots/ablation_bars.png', dpi=120)
plt.show()